In [17]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# =======================
# 1. LOAD DATA
# =======================
df = pd.read_csv(
    r"C:\\Users\\mimusaib\\Desktop\\Data_Science_Internship\\Week_4\\Dataset\\auto_pipeline_test.csv"
)

print("Initial shape:", df.shape)

# =======================
# 2. BASIC CLEANING
# =======================
df = df.drop_duplicates()

for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# =======================
# 3. IDENTIFY METADATA / ID COLUMNS (KEEP AS-IS)
# =======================
def get_metadata_columns(df):
    metadata_cols = []
    for col in df.columns:
        if any(key in col.lower() for key in ['id', 'name', 'email']):
            metadata_cols.append(col)
    return metadata_cols

metadata_cols = get_metadata_columns(df)
print("Metadata columns (kept as-is):", metadata_cols)

# =======================
# 4. FEATURE ENGINEERING
# =======================
if 'Age' in df.columns:
    df['Age_Group'] = pd.cut(
        df['Age'],
        bins=[0, 25, 45, 100],
        labels=['Young', 'Adult', 'Senior']
    )

# =======================
# 5. FEATURE TYPE DETECTION (EXCLUDING METADATA)
# =======================
def detect_feature_types(df, exclude_cols):
    work_df = df.drop(columns=exclude_cols)

    numeric = work_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical = work_df.select_dtypes(
        include=['object', 'category', 'string']
    ).columns.tolist()

    binary = [c for c in categorical if work_df[c].nunique() == 2]
    nominal = [c for c in categorical if work_df[c].nunique() > 2]

    return numeric, binary, nominal

num_features, bin_features, nom_features = detect_feature_types(df, metadata_cols)

# =======================
# 6. PREPROCESSING PIPELINE (PASSTHROUGH METADATA)
# =======================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('bin', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), bin_features),
        ('nom', OneHotEncoder(drop='first', handle_unknown='ignore'), nom_features)
    ],
    remainder='passthrough'   # 👈 keeps metadata columns unchanged
)

# =======================
# 7. APPLY & SAVE
# =======================
X_processed = preprocessor.fit_transform(df)
feature_names = preprocessor.get_feature_names_out()

processed_df = pd.DataFrame(X_processed, columns=feature_names)
processed_df.to_csv("final_ready_dataset_with_ids.csv", index=False)

print("\n✅ PREPROCESSING COMPLETE")
print("Final shape:", processed_df.shape)


Initial shape: (200, 7)
Metadata columns (kept as-is): []

✅ PREPROCESSING COMPLETE
Final shape: (200, 14)


In [18]:
print("Original columns:")
print(df.columns.tolist())

print("\nProcessed columns:")
print(feature_names)


Original columns:
['Age', 'Income', 'Experience_Years', 'Gender', 'City', 'Department', 'Performance_Level', 'Age_Group']

Processed columns:
['num__Age' 'num__Income' 'num__Experience_Years' 'bin__Gender_Male'
 'nom__City_Delhi' 'nom__City_Hyderabad' 'nom__City_Mumbai'
 'nom__Department_HR' 'nom__Department_IT' 'nom__Department_Marketing'
 'nom__Performance_Level_Low' 'nom__Performance_Level_Medium'
 'nom__Age_Group_Senior' 'nom__Age_Group_Young']
